In [ ]:
import csv
import random
import time
import json
from concurrent.futures import ThreadPoolExecutor

import requests
from bs4 import BeautifulSoup
from tqdm.notebook import tqdm


In [ ]:
URL = 'https://rgantd.kaisa.ru/type/SHUHOV?pageSize=1348'
SAVE_FILENAME = 'util_RGANTD_Shukhov_file_links.csv'

response = requests.get(URL, timeout=30)
soup = BeautifulSoup(response.text, 'html.parser')

descripton_buttons = soup.find_all('a', {'class': 'btn-description'}, href=True)
urls = []
for button in descripton_buttons:
    urls.append(button['href'].strip())

with open(SAVE_FILENAME, 'w', encoding='utf-8', newline='') as f:
    writer = csv.writer(f)
    for url in urls:
        writer.writerow([f'https://rgantd.kaisa.ru{url}'])


In [ ]:
SAVE_FILENAME = 'raw_data.json'
MAX_WORKERS = 5
USER_AGENTS = [
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:124.0) Gecko/20100101 Firefox/124.0',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36 Edg/123.0.2420.81',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36 OPR/109.0.0.0',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36',
]

def get_random_headers():
    return {
        'User-Agent': random.choice(USER_AGENTS)
    }

def delay(min_seconds=0.5, max_seconds=1.3):
    time.sleep(random.uniform(min_seconds, max_seconds))

def get_attributes(url: str) -> dict:
    try:
        delay()

        response = requests.get(url, headers=get_random_headers(), timeout=100)
        soup = BeautifulSoup(response.text, 'html.parser')

        attributes = soup.find_all('span', {'class': 'attribute-title'})
        
        attributes_dict = {}
        attributes_dict.update({'url': [url]})
        for attribute in attributes:
            attribute_title = attribute.get_text().strip()
            attribute_content = attribute.find_next_siblings()[1].get_text().strip()
            attributes_dict.setdefault(attribute_title, []).append(attribute_content)

        return attributes_dict
    
    except requests.exceptions.RequestException as e:
        print(f'Error fetching: {e}')

with open('util_RGANTD_Shukhov_file_links.csv', newline='') as f:
    reader = csv.reader(f)
    urls = list(reader)
urls = [url for list_url in urls for url in list_url]

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    files = list(tqdm(
        executor.map(get_attributes, urls),
        total=len(urls),
    ))

with open(SAVE_FILENAME, 'w', encoding='utf-8') as f:
    f.write(json.dumps(files, indent=2, ensure_ascii=False))
